### Import dependencies

In [2]:
import cv2
import numpy as np
from PIL import Image
import imagehash
from collections import deque
import os
from pathlib import Path

### Get non-consecutive non-empty frames from video (to be used for training YOLO)

In [3]:
def get_frames(
    input_file,
    output_dir,
    brightness_threshold=20,   # slightly higher since CLAHE is removed
    min_bright_pixels=500,
    hash_threshold=8,           # hamming distance for pHash diversity
    min_frame_gap=25,           # hard minimum frames between any two saves
):
    print('Opening:', input_file)
    print('Exists:', os.path.exists(input_file))

    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(input_file)

    saved_count = 0
    frame_index = 0
    last_saved_index = -min_frame_gap  # allow saving from frame 0
    last_hash = None
    frame_buffer = deque(maxlen=3)

    video_name = os.path.splitext(os.path.basename(input_file))[0]

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_index += 1

        # Detection pipeline — no CLAHE
        # Raw brightness is the right signal; CLAHE amplifies noise on dark frames
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)

        # Frame accumulation (keep — helps catch flickering organisms)
        frame_buffer.append(gray)
        accumulated = gray.copy()
        for old_frame in frame_buffer:
            accumulated = cv2.max(accumulated, old_frame)

        _, thresh = cv2.threshold(accumulated, brightness_threshold, 255, cv2.THRESH_BINARY)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

        if cv2.countNonZero(thresh) <= min_bright_pixels:
            continue  # empty frame, skip

        # Gate 1: hard minimum frame gap
        if frame_index - last_saved_index < min_frame_gap:
            continue

        # Gate 2: pHash diversity check — only save if visually different enough
        pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        current_hash = imagehash.phash(pil_img, hash_size=8)

        if last_hash is not None and (current_hash - last_hash) <= hash_threshold:
            continue  # too similar to last saved frame

        # Passed both gates — save
        filename = f"{video_name}_frame_{frame_index:05d}.jpg"
        cv2.imwrite(os.path.join(output_dir, filename), frame)
        last_hash = current_hash
        last_saved_index = frame_index
        saved_count += 1

    cap.release()
    print(f"Saved {saved_count} active frames from {frame_index} total frames.")
    return saved_count

Call function for a folder of videos

In [4]:
video_folder = Path("test_converted_videos/new")
video_paths = [video_folder / f for f in os.listdir(video_folder) if f != '.DS_Store']
for video_path in video_paths:
    get_frames(str(video_path), "active_frames")

### Use YOLO to pre-label active frames

In [ ]:
model = YOLO("runs/detect/train-8/weights/best.pt")
results = model.predict(
    source="active_frames",
    conf=0.15, iou=0.2,
    save_txt=True, save=False, device='mps'
)

### Get empty (dark) frames from videos (also to train YOLO)

In [20]:
def get_empty_frames(input_dir, max_frames=20):
    input_dir = Path(input_dir)
    output_dir = Path('empty_frames')

    cap = cv2.VideoCapture(str(input_dir))

    frame_buffer = deque(maxlen=3)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))

    saved = 0
    frame_index = 0

    while cap.isOpened() and saved < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        frame_index += 1

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = clahe.apply(gray)
        gray = cv2.GaussianBlur(gray, (5,5), 0)

        frame_buffer.append(gray)

        accumulated = gray.copy()
        for old_frame in frame_buffer:
            accumulated = cv2.max(accumulated, old_frame)

        _, thresh = cv2.threshold(accumulated, 15, 255, cv2.THRESH_BINARY)

        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

        # NOT active
        if cv2.countNonZero(thresh) <= 500 and frame_index % 50 == 0:
            cv2.imwrite(str(output_dir / f"empty_f{frame_index:03d}.jpg"),frame)
            saved += 1

    cap.release()
    print(f"Saved {saved} empty frames to {output_dir}")

In [23]:
os.makedirs('./empty_frames/', exist_ok=True)
video_dir = "test_converted_videos/new"
for file in os.listdir(video_dir):
    if file.startswith('.'):
        continue
    full_path = os.path.join(video_dir, file)
    get_empty_frames(full_path)

Saved 20 empty frames to empty_frames
Saved 20 empty frames to empty_frames
Saved 20 empty frames to empty_frames


Rename empty frames to background_xx.jpg

In [24]:
empty_dir = Path("./empty_frames")
images = sorted(empty_dir.glob("*.jpg"))
for i, img_path in enumerate(images):
    new_name = empty_dir / f"background_{i:03d}.jpg"
    img_path.rename(new_name)
    print(f"{img_path.name} -> {new_name.name}")

print(f"Renamed {len(images)} images.")

background_000.jpg -> background_000.jpg
background_001.jpg -> background_001.jpg
background_002.jpg -> background_002.jpg
background_003.jpg -> background_003.jpg
background_004.jpg -> background_004.jpg
background_005.jpg -> background_005.jpg
background_006.jpg -> background_006.jpg
background_007.jpg -> background_007.jpg
background_008.jpg -> background_008.jpg
background_009.jpg -> background_009.jpg
background_010.jpg -> background_010.jpg
background_011.jpg -> background_011.jpg
background_012.jpg -> background_012.jpg
background_013.jpg -> background_013.jpg
background_014.jpg -> background_014.jpg
background_015.jpg -> background_015.jpg
background_016.jpg -> background_016.jpg
background_017.jpg -> background_017.jpg
background_018.jpg -> background_018.jpg
background_019.jpg -> background_019.jpg
background_020.jpg -> background_020.jpg
background_021.jpg -> background_021.jpg
background_022.jpg -> background_022.jpg
background_023.jpg -> background_023.jpg
background_024.j

### Create empty label files (.txt) to be put inside YOLO's train folder (labels)
name of labels must match name of image --> background_xxx.jpg - background_xxx.txt

In [10]:
images_dir = Path.cwd()/'empty_frames'
labels_dir = Path.cwd()/'object-detection-project-8'/'train'/'labels'
bg_images = sorted(images_dir.glob("background_*.jpg"))
for img_path in bg_images:
    label_path = labels_dir /(img_path.stem + ".txt")
    label_path.touch()
    print(f"Created {label_path.name}")

print(f"Done - created {len(bg_images)} empty label files.")


Created background_000.txt
Created background_001.txt
Created background_002.txt
Created background_003.txt
Created background_004.txt
Created background_005.txt
Created background_006.txt
Created background_007.txt
Created background_008.txt
Created background_009.txt
Created background_010.txt
Created background_011.txt
Created background_012.txt
Created background_013.txt
Created background_014.txt
Created background_015.txt
Created background_016.txt
Created background_017.txt
Created background_018.txt
Created background_019.txt
Created background_020.txt
Created background_021.txt
Created background_022.txt
Created background_023.txt
Created background_024.txt
Created background_025.txt
Created background_026.txt
Created background_027.txt
Created background_028.txt
Created background_029.txt
Created background_030.txt
Created background_031.txt
Created background_032.txt
Created background_033.txt
Created background_034.txt
Created background_035.txt
Created background_036.txt
C

### Check that each image match with a label

In [29]:
train_images = set(p.stem for p in Path("/Users/alopias/Desktop/Huyen-deePi/object-detection-project-6/valid/images").glob("*.jpg"))
train_labels = set(p.stem for p in Path("/Users/alopias/Desktop/Huyen-deePi/object-detection-project-6/valid/labels").glob("*.txt"))

missing_labels = train_images - train_labels
missing_images = train_labels = train_images
print(f"Imags: {len(train_images)}, labels: {len(train_labels)}")
print(f"Missing labels: {missing_labels or 'None'}")
print(f"Missing images: {missing_images or 'None'}")

Imags: 139, labels: 139
Missing labels: None
Missing images: {'202308180848_frame_00735_jpg.rf.a350efaba504b647f527f107daf31e45', 'frame_01024_jpg.rf.524c7bfb46a7758c0b32b8131d313335', '202309241046_frame_01271_jpg.rf.d047cff51c200ac2105a087a4e8cb7fe', '202307250446_frame_00324_jpg.rf.0cf85acd12bb706cce18859a4457ae25', '202308070548_frame_00630_jpg.rf.f5133bdb48691d520ae504bc2bfcdc86', '202307251146_frame_00577_jpg.rf.4e562acce772787843f208cd3c9431e4', '202308201646_frame_00143_jpg.rf.b82bf65033e02b096c13319250f9dd45', '202308200046_frame_00323_jpg.rf.4a65e4bab5102c0e88848a7bd2a5dae2', '202310151646_frame_01217_jpg.rf.a7208e5c0d4b8ade3dca2caad53c975d', '202307031446_frame_01456_jpg.rf.b5a37c884ecb065e699a2b955a9cf6e8', '202307211646_frame_00027_jpg.rf.b0e3f8b0ea3a57c78b655919881c3694', 'frame_01681_jpg.rf.7dfa6fcf239f53d51962f1a0b5fc6585', '202308171748_frame_00064_jpg.rf.0435cc9269e5ccd7ca2617e2ff10ba00', 'background_095', '202307022346_frame_00839_jpg.rf.88ca853e8b7056bf18e59fe2b2f39